[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/templates/blob/main/machine-learning/notebooks/03_evaluation.ipynb)

# 03 — Final Evaluation

**Purpose.** Compare every model on the held-out test set, once, under
identical conditions.

**Inputs.** `data/processed/test.parquet` and the artifacts in `models/`.

**Outputs.** `reports/results/` and `reports/figures/`.

---

### Test-set discipline

This notebook is the **first and only** time the test split is read since
notebook 01 created it.

- Every model is scored on the same records, with the same metric function.
- Artifacts are loaded and used as-is: `transform`, never `fit`.
- If a result disappoints, the fix belongs in a `02x` notebook — and the test
  estimate is then compromised for every choice it informed. Tuning against the
  test set turns it into a second validation set, silently.

Run this once, when all models are final.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml`, the `models/`
lookup in section 3 and the `reports/` export in section 9 all resolve.
Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the few packages Colab does not
already ship. Note that `data/` and `models/` are DVC-tracked and therefore *not*
part of the clone — a fresh runtime has neither the frozen test split nor the
artifacts this notebook compares. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook. Forked the template? Change these three values
# and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/templates.git"
BRANCH = "main"
SUBDIR = "machine-learning"  # project root inside the repository

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1, and the two
# directories this notebook reads from and writes to. Defaults are the in-repo
# locations; the Drive cell below repoints them when running in Colab.
CONFIG_OVERRIDES: list[str] = []
MODEL_DIR = Path("models")
REPORT_DIR = Path("reports")

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# the same Drive layout notebooks 01 and 02a wrote to, so this notebook finds
# the frozen test split and the fitted artifacts.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# PROJECT_ROOT = Path("/content/drive/MyDrive/<project-name>")
# CONFIG_OVERRIDES += [f"test_path={PROJECT_ROOT}/data/processed/test.parquet"]
# MODEL_DIR = PROJECT_ROOT / "models"
# REPORT_DIR = PROJECT_ROOT / "reports"

## 1. Setup

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)  # empty unless section 0 filled it in
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Load the test split

The frozen one, written by notebook 01.

In [ ]:
from src.data.load import load_processed

test_df = load_processed(cfg, "test")
target = cfg.target
X_test, y_test = test_df.drop(columns=[target]), test_df[target]
print(f"{len(test_df):,} test records")

## 3. Load every model artifact

Each artifact carries its own fitted preprocessor, so the models can differ
completely inside while being called identically here.

In [ ]:
from src.utils.io import artifact_metadata, load_artifact

MODELS = ["model_a"]  # TODO: list every model to compare

paths = {name: MODEL_DIR / f"{name}.pkl" for name in MODELS}
artifacts = {name: load_artifact(path) for name, path in paths.items()}
{name: artifact_metadata(path) for name, path in paths.items()}

## 4. Predict — the same test set, the same call

The uniform loop is possible because every model implements the same interface.

In [ ]:
predictions = {}
for name, artifact in artifacts.items():
    X = artifact["preprocessor"].transform(X_test)   # transform, never fit
    predictions[name] = artifact["model"].predict(X)

{name: preds[:5] for name, preds in predictions.items()}

## 5. Results table

One metric function, applied identically to every model.

In [ ]:
from src.evaluation.metrics import PRIMARY_METRIC, evaluate

results = pd.DataFrame(
    {name: evaluate(y_test.to_numpy(), preds) for name, preds in predictions.items()}
).T
results.sort_values(PRIMARY_METRIC)

## 6. Comparison plots

Show the spread, not only the point estimate. A bar chart of single numbers
invites over-reading differences that sit inside the noise.

In [ ]:
from src.evaluation.analysis import comparison_plot

comparison_plot(results, PRIMARY_METRIC)

# TODO: bootstrap the test set to put an interval around each score, so the
#       ranking can be read honestly.

## 7. Error analysis

Where each model fails, and whether they fail on the same records. Models that
fail differently are candidates for an ensemble; models that fail identically
share a data problem.

In [ ]:
from src.evaluation.analysis import calibration_plot, error_table, residual_plot

for name, preds in predictions.items():
    print(f"--- {name} ---")
    display(error_table(y_test.to_numpy(), preds, features=X_test, top_n=10))

In [ ]:
# TODO: residual_plot (regression) or calibration_plot (classification) per model
# TODO: overlap of the worst-predicted records across models

## 8. Per-segment breakdown

An aggregate score can hide a model that is excellent on the common case and
unusable on the segment that matters.

In [ ]:
from src.evaluation.metrics import evaluate_by_group

# TODO: choose the segmentation that matters operationally — site, device,
#       class band, time window — and score each model within it.

## 9. Export

Everything a report or a decision meeting needs, written to disk rather than
left in notebook output.

In [ ]:
results_dir = REPORT_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(results_dir / "model_comparison.csv")

# TODO: save each figure to REPORT_DIR / "figures" at publication resolution
print(f"wrote {results_dir / 'model_comparison.csv'}")

## 10. Conclusions

State the decision and the reasoning, not just the winning row.

- **Selected model:** `<name>`
- **Why:** `<accuracy, and the cost / latency / interpretability trade-offs>`
- **Where it fails:** `<segments and error modes>`
- **Confidence:** `<is the gap to the runner-up larger than the noise?>`
- **Reproduced by:** commit `<hash>` + `dvc checkout`
- **Next steps:** `<what would improve it most>`